In [1]:
# ============ TEST SET CONCAT RUNS — FR + DE ============
import os, re, json
from datetime import datetime
from tqdm import tqdm
import torch
import numpy as np
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel, MarianMTModel, MarianTokenizer
from datasets import load_dataset, Dataset

def find_cache_dir():
    for c in ['baseline/cache', 'cache', '../baseline/cache']:
        if os.path.exists(c): return c
CACHE_DIR = find_cache_dir()

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
batch_size = 16
TOP_K = 1000
RUN_NAME = "TUWien-Track3-concat"
print(f"Device: {device}")

collection_embeddings = torch.load(os.path.join(CACHE_DIR, 'collection_embeddings.pt'))
collection_keys = np.load(os.path.join(CACHE_DIR, 'collection_keys.npy'))

# E5
e5_name = 'intfloat/multilingual-e5-large'
tok = AutoTokenizer.from_pretrained(e5_name)
model = AutoModel.from_pretrained(e5_name).to(device).eval()
print("E5 loaded.")

def avg_pool(h, m):
    h = h.masked_fill(~m[..., None].bool(), 0.0)
    return h.sum(dim=1) / m.sum(dim=1)[..., None]

def clean_tweet(t):
    t = re.sub(r'http\S+|www\.\S+', '', t)
    t = re.sub(r'@\w+', '', t)
    t = re.sub(r'#(\w+)', r'\1', t)
    t = re.sub(r'^\d+/n\s*Fact\s*\d+\s*:\s*', '', t, flags=re.IGNORECASE)
    t = re.sub(r'^\d+/\d+\s*', '', t)
    t = re.sub(r'[^\w\s\.\,\:\;\!\?\-\(\)\']+', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def translate(texts, src_lang, bs=32):
    mt_name = f"Helsinki-NLP/opus-mt-{src_lang}-en"
    mt_tok = MarianTokenizer.from_pretrained(mt_name)
    mt_m = MarianMTModel.from_pretrained(mt_name).to(device).eval()
    out = []
    for i in tqdm(range(0, len(texts), bs), desc=f"MT {src_lang}->en"):
        b = texts[i:i+bs]
        ins = mt_tok(b, return_tensors="pt", padding=True, truncation=True, max_length=256).to(device)
        with torch.no_grad():
            g = mt_m.generate(**ins, max_length=256, num_beams=1)
        out.extend(mt_tok.batch_decode(g, skip_special_tokens=True))
    del mt_m, mt_tok
    torch.mps.empty_cache() if device == 'mps' else None
    return out

def embed(texts, max_len=160):
    ds = Dataset.from_dict({"text": [f'query: {t}' for t in texts]})
    ds = ds.map(lambda e: tok(e["text"], max_length=max_len, truncation=True, padding="max_length", return_tensors="pt"),
                batched=True, remove_columns=ds.column_names).with_format("torch", columns=["input_ids", "attention_mask"])
    embs = []
    with torch.inference_mode():
        for b in tqdm(DataLoader(ds, batch_size=batch_size, num_workers=0), desc="Embed"):
            b = {k: v.to(device) for k, v in b.items()}
            embs.append(avg_pool(model(**b).last_hidden_state, b['attention_mask']).cpu())
    return F.normalize(torch.vstack(embs), p=2, dim=1)

def write_trec(out_path, indices, embs, run_name):
    cs = (embs @ collection_embeddings.T).numpy()
    top_idxs = np.argsort(-cs, axis=1)[:, :TOP_K]
    top_scores = np.take_along_axis(cs, top_idxs, axis=1)
    top_keys = collection_keys[top_idxs]
    with open(out_path, 'w') as f:
        for i, qid in enumerate(indices):
            for r in range(TOP_K):
                f.write(f"{qid} Q0 {top_keys[i,r]} {r+1} {top_scores[i,r]:.6f} {run_name}\n")
    print(f"Wrote {out_path}")

# Test sets
test_data = load_dataset("sschellhammer/CT26_Task1_SourceRetrievalForScientificWebClaims", "test")
os.makedirs("runs", exist_ok=True)

for lang in ["fr", "de"]:
    print(f"\n=== {lang.upper()} TEST CONCAT ===")
    split = test_data[lang]
    indices = list(split["index"])
    cleaned = [clean_tweet(t) for t in split["text"]]
    en_trans = translate(cleaned, lang)
    concat = [f"{c} {e}" for c, e in zip(cleaned, en_trans)]
    embs = embed(concat)
    write_trec(f"runs/{lang}_test_run_concat.txt", indices, embs, RUN_NAME)
    # save translations for archive
    with open(os.path.join(CACHE_DIR, f'test_en_{lang}.json'), 'w', encoding='utf-8') as f:
        json.dump(en_trans, f, ensure_ascii=False)

# top-100 versions for repo
for lang in ["fr", "de"]:
    src = f"runs/{lang}_test_run_concat.txt"
    dst = f"runs/{lang}_test_run_concat_top100.txt"
    with open(src) as fin, open(dst, 'w') as fout:
        prev = None; r = 0
        for line in fin:
            qid = line.split()[0]
            if qid != prev: prev = qid; r = 0
            r += 1
            if r <= 100: fout.write(line)
    sz = os.path.getsize(dst) / 1024 / 1024
    print(f"  {dst}: {sz:.1f} MB")

print("\nDone.")

/Users/ekinkeskin/air2026ss-checkthat-task1-checkthat-task1-methodology-3-group-1/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 7580.60it/s]


E5 loaded.

=== FR TEST CONCAT ===


/Users/ekinkeskin/air2026ss-checkthat-task1-checkthat-task1-methodology-3-group-1/.venv/lib/python3.11/site-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Embed: 100%|██████████| 77/77 [01:07<00:00,  1.14it/s]


Wrote runs/fr_test_run_concat.txt

=== DE TEST CONCAT ===


Embed: 100%|██████████| 55/55 [00:53<00:00,  1.03it/s]


Wrote runs/de_test_run_concat.txt
  runs/fr_test_run_concat_top100.txt: 5.2 MB
  runs/de_test_run_concat_top100.txt: 3.7 MB

Done.
